<a href="https://colab.research.google.com/github/rfroiman/DI_Bootcamp/blob/main/Week8/Day4/Vector_Databases_and_RAG_Chatbots.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: Vector Databases and RAG
Use this guided notebook and fill each TODO before running cells.

## What you'll learn
- Vector search strategies (KNN, ANN) and evaluation.
- Vector database utility (similarity search, RAG).
- Differences between vector DBs, libraries, and plugins.
- Best practices for vector store usage and performance.
- How LMs use context; embedding generation and storage.
- Querying vector stores and applying LMs for QA with retrieved context.

## What you'll build
A functional RAG pipeline with FAISS and ChromaDB, plus QA over retrieved context using a Hugging Face model.

## 0. Setup
Run the install cell once. If your platform needs system deps (e.g., libomp for FAISS), follow instructions in comments.

In [ ]:
%pip uninstall -y pydantic-core pydantic
%pip install -U "pydantic<2"
%pip install -U "faiss-cpu>=1.8.0" "chromadb==0.3.21"
%pip install -U "numpy<2" sentence-transformers transformers

In [1]:
%pip install -q -U faiss-cpu chromadb sentence-transformers transformers accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7

In [1]:
import numpy as np
import pandas as pd
import faiss
import json
import chromadb

from sentence_transformers import SentenceTransformer, InputExample
from chromadb.config import Settings
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

print("All libraries imported successfully!")

All libraries imported successfully!


## 🌟 Exercise 1 · Data loading and preparation

In [4]:
import io
import zipfile
import requests
import pandas as pd

dataset_url = "https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/Week%205/Day%204%20-%20Diving%20Deep%20into%20Vector%20Databases%20and%20RAG%20Chatbots/labelled_newscatcher_dataset.zip"

response = requests.get(dataset_url, timeout=60)
response.raise_for_status()

with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
    with archive.open("labelled_newscatcher_dataset.csv") as file:
        pdf = pd.read_csv(file, sep=";")

print("Dataset shape:", pdf.shape)
display(pdf.head())

Dataset shape: (108774, 6)


,topic,link,domain,published_date,title,lang
0,SCIENCE,https://www.eurekalert.org/pub_releases/2020-0...,eurekalert.org,2020-08-06 13:59:45,A closer look at water-splitting's solar fuel ...,en
1,SCIENCE,https://www.pulse.ng/news/world/an-irresistibl...,pulse.ng,2020-08-12 15:14:19,"An irresistible scent makes locusts swarm, stu...",en
2,SCIENCE,https://www.express.co.uk/news/science/1322607...,express.co.uk,2020-08-13 21:01:00,Artificial intelligence warning: AI will know ...,en
3,SCIENCE,https://www.ndtv.com/world-news/glaciers-could...,ndtv.com,2020-08-03 22:18:26,Glaciers Could Have Sculpted Mars Valleys: Study,en
4,SCIENCE,https://www.thesun.ie/tech/5742187/perseid-met...,thesun.ie,2020-08-12 19:54:36,Perseid meteor shower 2020: What time and how ...,en


In [5]:
# Create a unique identifier for each article
pdf["id"] = range(len(pdf))

# Create a subset of 1,000 articles
pdf_subset = pdf.head(1000).copy()

# Check the results
print("Full dataset:", pdf.shape)
print("Subset:", pdf_subset.shape)

display(pdf_subset.head())

Full dataset: (108774, 7)
Subset: (1000, 7)


,topic,link,domain,published_date,title,lang,id
0,SCIENCE,https://www.eurekalert.org/pub_releases/2020-0...,eurekalert.org,2020-08-06 13:59:45,A closer look at water-splitting's solar fuel ...,en,0
1,SCIENCE,https://www.pulse.ng/news/world/an-irresistibl...,pulse.ng,2020-08-12 15:14:19,"An irresistible scent makes locusts swarm, stu...",en,1
2,SCIENCE,https://www.express.co.uk/news/science/1322607...,express.co.uk,2020-08-13 21:01:00,Artificial intelligence warning: AI will know ...,en,2
3,SCIENCE,https://www.ndtv.com/world-news/glaciers-could...,ndtv.com,2020-08-03 22:18:26,Glaciers Could Have Sculpted Mars Valleys: Study,en,3
4,SCIENCE,https://www.thesun.ie/tech/5742187/perseid-met...,thesun.ie,2020-08-12 19:54:36,Perseid meteor shower 2020: What time and how ...,en,4


## 🌟 Exercise 2 · Vectorization with Sentence Transformers

In [6]:
from sentence_transformers import InputExample

# Create an InputExample for each news title
def example_create_fn(title: str) -> InputExample:
    return InputExample(texts=[title])

# Convert the titles into InputExample objects
faiss_train_examples = pdf_subset["title"].fillna("").apply(
    example_create_fn
).tolist()

# Display the first 10 examples
faiss_train_examples[:10]

In [7]:
from sentence_transformers import SentenceTransformer

# Load the pre-trained embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [8]:
# Extract news titles
titles_list = pdf_subset["title"].fillna("").tolist()

# Generate embeddings for all titles
faiss_title_embedding = model.encode(
    titles_list,
    show_progress_bar=True
)

# Check embedding dimensions
print("Number of embeddings:", len(faiss_title_embedding))
print("Embedding dimensions:", len(faiss_title_embedding[0]))

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Number of embeddings: 1000
Embedding dimensions: 384


## 🌟 Exercise 3 · FAISS indexing and search

In [9]:
import numpy as np
import faiss

# Prepare the dataset and article IDs
pdf_to_index = pdf_subset.copy()
id_index = pdf_to_index["id"].to_numpy(dtype=np.int64)

# Convert embeddings to float32
content_encoded_normalized = np.array(
    faiss_title_embedding,
    dtype=np.float32
).copy()

# Normalize vectors for cosine similarity
faiss.normalize_L2(content_encoded_normalized)

print("Vectors shape:", content_encoded_normalized.shape)
print("IDs shape:", id_index.shape)

Vectors shape: (1000, 384)
IDs shape: (1000,)


In [10]:
# Create a FAISS index using inner product similarity
index_content = faiss.IndexIDMap(
    faiss.IndexFlatIP(content_encoded_normalized.shape[1])
)

# Add normalized vectors and their IDs
index_content.add_with_ids(
    content_encoded_normalized,
    id_index
)

print("Total indexed vectors:", index_content.ntotal)

Total indexed vectors: 1000


In [11]:
def search_content(query, pdf_to_index, k=3):
    # Convert the query into an embedding
    query_vector = model.encode(
        [query],
        convert_to_numpy=True
    ).astype(np.float32)

    # Normalize the query vector
    faiss.normalize_L2(query_vector)

    # Search for the most similar articles
    similarities, ids = index_content.search(query_vector, k)

    # Retrieve matching articles
    results = pdf_to_index.set_index("id").loc[ids[0]].copy()

    # Add similarity scores
    results["similarities"] = similarities[0]

    return results.reset_index()


# Test the search function
display(search_content("animal", pdf_to_index, k=5))

,id,topic,link,domain,published_date,title,lang,similarities
0,176,TECHNOLOGY,https://www.pushsquare.com/news/2020/08/random...,pushsquare.com,2020-08-03 16:30:00,Random: You Can Pick Up and Pet Cats in Assass...,en,0.391902
1,975,HEALTH,https://www.news-medical.net/news/20200813/Res...,news-medical.net,2020-08-13 05:18:00,Researchers explore social behavior of animals...,en,0.376784
2,99,TECHNOLOGY,https://www.gematsu.com/2020/08/ghostwire-toky...,gematsu.com,2020-08-07 16:43:13,Ghostwire: Tokyo confirms dog petting,en,0.344059
3,928,SCIENCE,https://www.thecut.com/2020/08/scientists-say-...,thecut.com,2020-08-04 12:52:00,Just Let This Lizard Be a Dinosaur,en,0.317387
4,762,SCIENCE,https://af.reuters.com/article/worldNews/idAFK...,af.reuters.com,2020-08-13 16:51:00,'Secret' life of sharks: Study reveals their s...,en,0.295497


## 🌟 Exercise 4 · ChromaDB collection and querying

In [13]:
import chromadb

# Initialize ChromaDB client
chroma_client = chromadb.Client()

import chromadb

# Initialize ChromaDB client
chroma_client = chromadb.Client()

# Define collection name
collection_name = "my_news"

# Create the collection or retrieve it if it already exists
collection = chroma_client.get_or_create_collection(
    name=collection_name
)

print("Collection created:", collection.name)

Collection created: my_news


In [14]:
# Select the first 100 news articles
news_subset = pdf_subset.head(100)

# Add documents, metadata, and unique IDs to ChromaDB
collection.add(
    documents=news_subset["title"].fillna("").tolist(),
    metadatas=[
        {"topic": str(topic)}
        for topic in news_subset["topic"].fillna("UNKNOWN")
    ],
    ids=news_subset["id"].astype(str).tolist()
)

print("Documents stored:", collection.count())

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 54.5MiB/s]


Documents stored: 100


In [15]:
import json

# Search for the 10 most relevant news articles
results = collection.query(
    query_texts=["space"],
    n_results=10
)

# Display the search results
print(json.dumps(results, indent=4))

{
    "ids": [
        [
            "72",
            "7",
            "30",
            "26",
            "23",
            "76",
            "69",
            "40",
            "47",
            "75"
        ]
    ],
    "embeddings": null,
    "documents": [
        [
            "Beck teams up with NASA and AI for 'Hyperspace' visual album experience",
            "Orbital space tourism set for rebirth in 2021",
            "NASA drops \"insensitive\" nicknames for cosmic objects",
            "\u2018It came alive:\u2019 NASA astronauts describe experiencing splashdown in SpaceX Dragon",
            "Hubble Uses Moon As \u201cMirror\u201d to Study Earth\u2019s Atmosphere \u2013 Proxy in Search of Potentially Habitable Planets Around Other Stars",
            "Australia's small yet crucial part in the mission to find life on Mars",
            "NASA Astronauts in SpaceX Capsule Splashdown in Gulf Of Mexico",
            "SpaceX's Starship spacecraft saw 150 meters high",
          

## 🌟 Exercise 5 · Question answering with a Hugging Face model

In [16]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Select a lightweight language model
model_id = "distilgpt2"

# Load tokenizer and language model
tokenizer = AutoTokenizer.from_pretrained(model_id)
lm_model = AutoModelForCausalLM.from_pretrained(model_id)

print("Language model loaded successfully!")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Language model loaded successfully!


In [17]:
# Create the text generation pipeline
pipe = pipeline(
    "text-generation",
    model=lm_model,
    tokenizer=tokenizer,
    max_new_tokens=150,
    device_map="auto"
)

print("Text generation pipeline created successfully!")

Text generation pipeline created successfully!


In [18]:
# Define the user's question
question = "What are the latest developments in space exploration?"

# Retrieve relevant documents from ChromaDB
results = collection.query(
    query_texts=[question],
    n_results=3
)

# Combine retrieved documents into context
context = "\n".join(results["documents"][0])

# Create the prompt
prompt_template = f"""Relevant context:
{context}

Question: {question}
Answer:"""

print(prompt_template)

Relevant context:
Orbital space tourism set for rebirth in 2021
Beck teams up with NASA and AI for 'Hyperspace' visual album experience
Australia's small yet crucial part in the mission to find life on Mars

Question: What are the latest developments in space exploration?
Answer:


In [19]:
# Generate an answer using the retrieved context
lm_response = pipe(
    prompt_template,
    max_new_tokens=100,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id
)

# Display the generated answer
print(lm_response[0]["generated_text"])

Relevant context:
Orbital space tourism set for rebirth in 2021
Beck teams up with NASA and AI for 'Hyperspace' visual album experience
Australia's small yet crucial part in the mission to find life on Mars

Question: What are the latest developments in space exploration?
Answer:
The first major milestone in the mission is the launch of the first manned space station, the first manned space station. The first manned space station, the first manned space station, was launched in 2006. The first manned space station, the first manned space station, was launched in 2006. The first manned space station, the first manned space station, was launched in 2006. The first manned space station, the first manned space station, was launched in 2006. The first manned space station, the first


In [20]:
# Define a more specific question
question = "What is the news about orbital space tourism?"

# Retrieve relevant documents
results = collection.query(
    query_texts=[question],
    n_results=2
)

# Build the context
context = "\n".join(results["documents"][0])

# Create a more restrictive prompt
prompt_template = f"""Use only the following news headlines.

Context:
{context}

Question: {question}
Answer:"""

# Generate a shorter response
lm_response = pipe(
    prompt_template,
    max_new_tokens=50,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id
)

print(lm_response[0]["generated_text"])

Use only the following news headlines.

Context:
Orbital space tourism set for rebirth in 2021
NASA Astronauts in SpaceX Capsule Splashdown in Gulf Of Mexico

Question: What is the news about orbital space tourism?
Answer:
Space tourism is a new concept that has been developed by NASA and NASA. It is a new concept that has been developed by NASA and NASA. It is a new concept that has been developed by NASA and NASA. It is a new concept that


### Analysis: Prompt and Context Experimentation

The RAG pipeline successfully retrieved relevant news headlines from ChromaDB and provided them as context to the language model.

In the first experiment, the model generated repetitive and inaccurate information about space exploration.

In the second experiment, a more specific question, fewer retrieved documents, and a shorter response limit were used. However, the model continued to repeat phrases and generate unsupported information.

These results demonstrate that retrieval alone does not guarantee accurate answers. The quality of the language model is also essential for a reliable RAG system.

DistilGPT-2 is suitable for demonstrating text generation, but it is not instruction-tuned or optimized for question answering.